# Regression Active Learning Acquisition Workflow

BOではなく「何を効率よく学ぶか」をdecision purposeにした回帰Active Learningを、posterior uncertaintyからcandidate生成・観測追加まで接続します。

関連理論:

- [Active Learning](../../docs/theory/acquisition/09_active_learning.md)
- [Level-set](../../docs/theory/acquisition/10_level_set.md)

```text
surrogate posterior
    ↓
learning targetを定義
    ↓
Variance / EPIG / qNIPV / Straddle
    ↓
candidate selection
    ↓
observation
    ↓
posterior update
```

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.active_learning import qNegIntegratedPosteriorVariance
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from robotorchan.acquisition import (
    ExpectedPredictiveInformationGain,
    PosteriorVariance,
    Straddle,
)
from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Sparse regression observations

function learningが目的なので、objectiveの最大値はdecision criterionに使いません。

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * X


train_X = torch.tensor([[0.05], [0.22], [0.48], [0.72], [0.95]])
train_Y = response(train_X)
bounds = torch.tensor([[0.0], [1.0]])

model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()

## 3. Posterior uncertainty

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
mean = posterior.mean.squeeze(-1)
std = posterior.variance.clamp_min(0.0).sqrt().squeeze(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid.squeeze(-1), mean, label="posterior mean")
ax.fill_between(grid.squeeze(-1), mean - 2 * std, mean + 2 * std, alpha=0.2)
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
ax.legend()
plt.show()

## 4. Pointwise function learning: PosteriorVariance

`PosteriorVariance`はcandidate自身のmarginal epistemic uncertaintyを基準にするq=1 criterionです。

In [ ]:
variance_acq = PosteriorVariance(model)
variance_candidate, variance_value = optimize_acqf(
    acq_function=variance_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("variance candidate:", variance_candidate)
print("variance score:", variance_value)

## 5. Prediction-oriented learning: EPIG

EPIGはcandidate自身が不確実かではなく、candidate観測が**target inputsでのpredictions**についてどれだけ情報を与えるかを評価します。ここではdeploymentが中央領域に集中すると仮定します。

In [ ]:
target_X = torch.linspace(0.35, 0.65, 31).unsqueeze(-1)
epig = ExpectedPredictiveInformationGain(
    model,
    target_X,
    observation_noise=1e-4,
)
epig_candidate, epig_value = optimize_acqf(
    acq_function=epig,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("EPIG candidate:", epig_candidate)
print("EPIG score:", epig_value)

## 6. Batch target-region learning: qNIPV

BoTorch native `qNegIntegratedPosteriorVariance`はintegration points全体で、batch観測後に残るposterior varianceを評価します。pointwise variance上位2点を取る方法ではありません。

In [ ]:
integration_X = torch.linspace(0.0, 1.0, 41).unsqueeze(-1)
nipv = qNegIntegratedPosteriorVariance(model=model, mc_points=integration_X)
nipv_candidates, nipv_value = optimize_acqf(
    acq_function=nipv,
    bounds=bounds,
    q=2,
    num_restarts=3,
    raw_samples=32,
)
assert nipv_candidates.shape == torch.Size([2, 1])
print("qNIPV joint candidates:", nipv_candidates.squeeze(-1))
print("qNIPV value:", nipv_value)

## 7. Boundary learning is a different AL objective

function全体ではなく`f(x)=target`の境界を学ぶならStraddleを使えます。uncertaintyだけでなくposterior meanのtarget proximityもcriterionに入ります。

In [ ]:
target_level = 0.0
straddle = Straddle(model, target=target_level, beta=1.96)
boundary_candidate, boundary_value = optimize_acqf(
    acq_function=straddle,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("Straddle candidate:", boundary_candidate)
print("Straddle score:", boundary_value)

## 8. Candidate locations reflect different learning targets

acquisition valueの絶対値はcriterion間で比較しません。Variance、EPIG、qNIPV、Straddleは学習したいquantity自体が異なります。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), response(grid).squeeze(-1), label="true response")
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
for name, candidate in {
    "Variance": variance_candidate,
    "EPIG": epig_candidate,
    "Straddle": boundary_candidate,
}.items():
    ax.scatter(
        candidate.item(),
        response(candidate).item(),
        marker="x",
        s=80,
        label=name,
    )
for index, candidate in enumerate(nipv_candidates):
    ax.scatter(
        candidate.item(),
        response(candidate.unsqueeze(0)).item(),
        marker="+",
        s=90,
        label="qNIPV batch" if index == 0 else None,
    )
ax.legend(ncol=2)
plt.show()

## 9. Observe one AL candidate and update the posterior

代表としてprediction-orientedなEPIG candidateを観測し、dataへ追加してrefitします。

In [ ]:
new_Y = response(epig_candidate)
updated_X = torch.cat([train_X, epig_candidate], dim=0)
updated_Y = torch.cat([train_Y, new_Y], dim=0)

updated_model = SingleTaskGP(updated_X, updated_Y)
fit_gpytorch_mll(updated_model.make_mll())
updated_model.eval()

with torch.no_grad():
    before = model.posterior(target_X).variance.mean()
    after = updated_model.posterior(target_X).variance.mean()
print("mean target variance before:", before.item())
print("mean target variance after:", after.item())

## 10. Decision-purpose map

| Criterion | Learning target | Candidate semantics |
| --- | --- | --- |
| PosteriorVariance | candidateのlatent function | q=1 marginal uncertainty |
| EPIG | target-set predictions | q=1 predictive information gain |
| qNIPV | integration region | joint batch variance reduction |
| Straddle | specified response boundary | q=1 uncertainty + target proximity |

BOのEI/UCBのように「良いobjective valueを見つける」ことは、このNotebookの最終目的ではありません。

## 11. Important boundaries

- predictive varianceとreducible epistemic uncertaintyを無条件に同一視しません。
- EPIGのcurrent contractはq=1、single-output、joint Gaussian posterior、finite `target_X`です。
- current `PosteriorVariance/Std` runtimeはensemble posteriorをrejectします。
- qNIPVはBoTorch native pathを直接使い、robotorchan wrapperを追加しません。
- Straddleのpointwise上位q点はjoint batch level-set acquisitionではありません。
- classification-specific entropy/BALD/margin acquisitionは現在のregression workflowへ混ぜません。
- MESはoptimum valueについてのinformation gainであり、prediction-oriented EPIGとはinformation targetが違います。

## 12. Related components

- `docs/optimization/regression_active_learning.md`: regression AL runtime guide
- `docs/optimization/epig.md`: EPIG contract
- `docs/theory/acquisition/05_information_theoretic.md`: BO側information target
- `docs/theory/acquisition/10_level_set.md`: boundary learning

Active Learningでは最初に「information / uncertainty about what?」を決めてからacquisitionを選びます。